# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
RUN_TO_ANALYSE = "4"
CP_DIR = Path("output") / "runs" / str(RUN_TO_ANALYSE)

df = pd.read_csv(CP_DIR / "combined_cp_metrics.csv")

In [ ]:
cols = ['coverage', 'cov_frau1', 'cov_nongerman', 'cov_nongerman_male', 'cov_nongerman_female']

In [ ]:
df_sub = df[cols]

In [ ]:
df_sub

In [ ]:
cov_subgroups_range = 0.75, 1.0

fig, ax = plt.subplots(figsize=(8, 6))

# Positions for each box
positions = np.arange(1, len(cols) + 1)

# Draw the boxplots
bp = ax.boxplot(
    [df_sub[col] for col in cols],
    positions=positions,
    widths=0.6,
    patch_artist=True,
    notch=True
)

colors = ["darkgray", "#CAB2D6", "lightskyblue", "aquamarine", "#FFFF99",]

# Style the boxes and medians
for patch, median, color in zip(bp["boxes"], bp["medians"], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor("black")
    patch.set_linewidth(1.5)
    median.set_color("black")
    median.set_linewidth(2)

# Style whiskers and caps
for whisker in bp["whiskers"]:
    whisker.set_color("black")
    whisker.set_linewidth(1.5)
for cap in bp["caps"]:
    cap.set_color("black")
    cap.set_linewidth(1.5)

# Overlay the actual data points with a little horizontal jitter
for i, col in enumerate(cols):
    y = df_sub[col]
    x = np.random.normal(positions[i], 0.04, size=len(y))
    ax.scatter(
        x,
        y,
        alpha=0.7,
        color=colors[i],
        edgecolors="black",
        linewidths=0.5,
        s=40
    )

# Add a horizontal line at coverage = 0.90
ax.axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='Target coverage = 0.90')
# Add a light grid
ax.yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.7)

new_labels = [
    "overall",
    "female",
    "non-German",
    "non-German male",
    "non-German female"
]
ax.set_xticks(positions)
ax.set_xticklabels(new_labels, rotation=30)

ax.set_ylabel("Group-Conditional Coverage", fontsize=20)
ax.set_ylim(cov_subgroups_range)

ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)

plt.tight_layout()

output_path = CP_DIR / "multi-coverage_subgroups.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Medians for subgroups
df_sub[cols].median().tolist()

In [ ]:
# Plot histogram
plt.figure(figsize=(7, 4))
plt.hist(df["avg_size"], bins=30, color='aquamarine', edgecolor='black')

plt.tick_params(axis='both', which='major', labelsize=16)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.xlabel("Average Prediction Set Size", fontsize=18)
plt.ylabel("Count", fontsize=18)
plt.grid(False)
plt.tight_layout()

output_path = CP_DIR / "multi-avg_size_distribution.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
max_value = df["avg_size"].max()
min_value = df["avg_size"].min()

print("Max avg_size:", max_value)
print("Min avg_size:", min_value)

## Fractions of Ambiguous Predictions

In [ ]:
# Select all columns starting with frac_ambiguous_

ambiguous_cols = ["fraction_ambiguous"] + [col for col in df.columns if col.startswith("frac_ambiguous_")]
df_amb = df[ambiguous_cols]
df_amb

In [ ]:
# Prepare data for seaborn (needs to be in long format)
df_plot = df_amb.melt(var_name='subgroup', value_name='fraction_ambiguous')

# Column mapping for better labels
col_mapping = {
    "fraction_ambiguous": "overall",
    "frac_ambiguous_frau1": "female",
    "frac_ambiguous_nongerman": "non-German",
    "frac_ambiguous_nongerman_male": "non-German male",
    "frac_ambiguous_nongerman_female": "non-German female"
}

# Apply mapping to the subgroup names
df_plot['subgroup_label'] = df_plot['subgroup'].map(col_mapping)

# Define order to match our preference
subgroup_order = col_mapping.values()

# Colors for each subgroup
colors = ["darkgray", "#CAB2D6", "lightskyblue", "aquamarine", "#FFFF99",]
# colors = ["#1d2f57", "#6a0008", "#c34b02", "#f8ae27", "#7b7609"]

# Set seaborn style
sns.set(style="whitegrid", palette="pastel", font_scale=2.25)

# Create horizontal violin plot
fig, ax = plt.subplots(figsize=(10, 6))

# Draw horizontal violin plots using seaborn
sns.violinplot(
    data=df_plot,
    y='subgroup_label',
    x='fraction_ambiguous',
    ax=ax,
    inner='box',
    cut=0,
    order=subgroup_order,
    palette=colors
)

# Overlay median dots
medians = df_plot.groupby('subgroup_label')['fraction_ambiguous'].median()
for i, subgroup in enumerate(subgroup_order):
    if subgroup in medians:
        ax.plot(medians[subgroup], i, color='lemonchiffon', marker='o', markersize=8, zorder=3)

# Set labels
ax.set_xlabel("Fraction Ambiguous")
ax.set_ylabel("")

ax.tick_params(axis='both')

plt.tight_layout()

output_path = CP_DIR / "multi-frac_ambiguous.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()